In [150]:
import requests
import httpx

EMBEDDING_URL = "http://localhost:8100/embed"

In [151]:
def get_error_list(date: str):
    url = "http://10.122.100.172:9000/analyze/error/list"

    payload = {
        "date": date
    }

    resp = requests.post(url, json=payload)
    return resp.json()

In [152]:
def get_error_file(date: str, file_list: list):
    url = "http://10.122.100.172:9000/analyze/error/file"

    result = []
    for file in file_list :
        
        payload = {
            "date": date,
            "filename" : file
        }

        resp = requests.post(url, json=payload)
        result.append(resp.json())
    return result

In [ ]:
date = "2026-03-31"
file_list = get_error_list(date).get("files")
files = get_error_file(date, file_list)

In [263]:
def format_action_plan(action_plan: list[str]) -> str:
    return "\n".join(
        [f"{i+1}. {item}" for i, item in enumerate(action_plan)]
    )

In [264]:
def build_raw_text(file):
    cause_list = file["output_json"]["causeList"]

    return "\n\n".join(
        f"""[CAUSE {i+1}]
title : {c["title"]}
cause : {c["cause"]}
evidence : {c["evidence"]}
actionPlan :
{format_action_plan(c["actionPlan"])}
"""
        for i, c in enumerate(cause_list)
    )

In [265]:
async def embed_texts(texts: list[str]) -> list[list[float]]:
    async with httpx.AsyncClient(timeout=5.0) as client:
        resp = await client.post(
            EMBEDDING_URL,
            json={"texts": texts}
        )
        resp.raise_for_status()
        return resp.json()["dense_vecs"]

In [266]:
def format_action_plan(action_plan: list[str]) -> str:
    return "\n".join(f"{i+1}. {item}" for i, item in enumerate(action_plan))


def build_search_doc(file: dict, date: str, filename: str) -> dict:
    cause_list = file.get("output_json", {}).get("causeList", [])

    sections = []
    keyword_parts = []

    for i, cause in enumerate(cause_list, start=1):
        title = cause.get("title", "")
        cause_text = cause.get("cause", "")
        evidence = cause.get("evidence", "")
        action_plan = cause.get("actionPlan", [])

        # 의미 검색용 본문
        sections.append(
            f"""[CAUSE {i}]
title: {title}
cause: {cause_text}
actionPlan:
{format_action_plan(action_plan)}"""
        )

        # 키워드 검색용
        # evidence는 여기로 몰아넣기
        keyword_parts.append(title)
        keyword_parts.append(evidence)

    vector_text = "\n\n".join(sections).strip()
    keywords = " ".join(x for x in keyword_parts if x).strip()

    return {
        "date": date,
        "filename": filename,
        "request_id": file.get("request_id"),
        "timestamp": file.get("timestamp"),
        "model_name": file.get("model_name"),
        "error_message": file.get("error_message"),

        "vector_text": vector_text,
        "keywords": keywords,
    }

In [267]:
INDEX_NAME = "error"

search_docs = [
    build_search_doc(file=f, date=date, filename=file_list[i])
    for i, f in enumerate(files)
]

vector_text_list = [doc["vector_text"] for doc in search_docs]
embeddings = await embed_texts(vector_text_list)

final_docs = []
for doc, emb in zip(search_docs, embeddings):
    doc["vector"] = emb
    final_docs.append({
        "_op_type": "index",
        "_index": INDEX_NAME,
        "_id": doc["filename"],
        "_source": doc })

In [268]:
def bulk_insert(docs: list[dict]) -> None:
    from opensearchpy.helpers import bulk
    bulk(client, docs)
    print(f"Inserted {len(docs)} docs")

In [269]:
bulk_insert(final_docs)

Inserted 27 docs


In [285]:
query = ["NoHandlerFoundException No handler found for GET /favicon.ico"]
query_vector = await embed_texts(query)

In [286]:
def search_similar(query: list, query_vector: list, k: int = 3):
    
    body = {
        "size": k,
        "query": {
            "hybrid": {
                "queries": [
                    {
                        "match": {
                            "keyword": query[0]
                        }
                    },
                    {
                        "knn": {
                            "vector": {
                                "vector": query_vector[0],
                                "k": k
                            }
                        }
                    }
                ]
            }
        }
    }

    resp = client.search(index=INDEX_NAME, body=body)
    return resp["hits"]["hits"]

In [287]:
results = search_similar(query, query_vector)

In [288]:
for hit in results:
    print(hit["_score"], hit["_source"]["keywords"])

-9549512000.0 favicon.ico 요청에 대한 핸들러 미설정 NoHandlerFoundException No handler found for GET /favicon.ico
-4422440400.0 favicon.ico 요청에 대한 핸들러 미설정 NoHandlerFoundException No handler found for GET /favicon.ico
-4422440400.0 favicon.ico 요청에 대한 핸들러 미설정 NoHandlerFoundException No handler found for GET /favicon.ico


In [289]:
def make_rerank_text(doc: dict) -> str:
    text = "\n".join([
        str(doc.get("keywords", "") or ""),
        str(doc.get("vector_text", "") or ""),
        str(doc.get("content", "") or "")
    ]).strip()
    return text

In [290]:
import httpx

RERANK_URL = "http://localhost:8100/rerank"


async def rerank_documents(query: str, docs: list[dict], top_n: int = 5) -> list[dict]:
    if not docs:
        return []

    passages = [make_rerank_text(doc) for doc in docs]

    print("docs:", len(docs))
    print("passages:", len(passages))

    async with httpx.AsyncClient(timeout=10.0) as client:
        resp = await client.post(
            RERANK_URL,
            json={
                "query": str(query),
                "passages": passages,
                "max_length": 512,
            },
        )

        print(resp.status_code)
        print(resp.text)

        resp.raise_for_status()
        scores = resp.json()["scores"]

    scored_docs = []
    for doc, score in zip(docs, scores):
        item = dict(doc)
        item["rerank_score"] = float(score)
        scored_docs.append(item)

    scored_docs.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored_docs[:top_n]

In [ ]:
reranked = await rerank_documents(query[0], [doc["_source"] for doc in results], top_n=5)

for doc in reranked:
    rag_doc = get_error_file(doc['date'], [doc["filename"]])[0]
    rag_doc['rerank_score'] = doc['rerank_score']
    print(rag_doc)

docs: 3
passages: 3
200
{"scores":[8.239623069763184,8.239623069763184,8.239623069763184]}


In [298]:
for doc in reranked:
    rag_doc = get_error_file(doc['date'], [doc["filename"]])[0]
    rag_doc['rerank_score'] = doc['rerank_score']
    print(rag_doc)

{'request_id': '4bc70337', 'timestamp': '2026-04-02T20:20:44.423351+09:00', 'client_ip': '10.101.5.160', 'client_port': 36810, 'model_name': 'gpt-oss-20b', 'prompt_tokens': 1954, 'completion_tokens': 623, 'total_tokens': 2577, 'elapsed_sec': 7.415, 'input_text': {'message': 'org.springframework.web.servlet.NoHandlerFoundException: No handler found for GET /favicon.ico\n\tat org.springframework.web.servlet.DispatcherServlet.noHandlerFound(DispatcherServlet.java:1287)\n\tat org.springframework.web.servlet.DispatcherServlet.doDispatch(DispatcherServlet.java:1050)\n\tat org.springframework.web.servlet.DispatcherServlet.doService(DispatcherServlet.java:965)\n\tat org.springframework.web.servlet.FrameworkServlet.processRequest(FrameworkServlet.java:1006)\n\tat org.springframework.web.servlet.FrameworkServlet.doGet(FrameworkServlet.java:898)\n\tat javax.servlet.http.HttpServlet.service(HttpServlet.java:529)\n\tat org.springframework.web.servlet.FrameworkServlet.service(FrameworkServlet.java:8

In [ ]:
rag_doc

{'date': '2026-04-02',
 'filename': '20260402_202037_4bc70337',
 'request_id': '4bc70337',
 'timestamp': '2026-04-02T20:20:44.423351+09:00',
 'model_name': 'gpt-oss-20b',
 'error_message': None,
 'vector_text': '[CAUSE 1]\ntitle: favicon.ico 요청에 대한 핸들러 미설정\ncause: GET /favicon.ico 요청이 들어왔지만, Spring MVC에서 해당 경로를 처리할 핸들러가 없어서 NoHandlerFoundException이 발생했습니다. 이는 보통 정적 리소스 매핑이 없거나, favicon.ico 파일이 존재하지 않을 때 발생합니다.\nactionPlan:\n1. 정적 리소스 매핑을 확인하고, /favicon.ico 경로가 존재하도록 설정합니다.\n2. src/main/resources/static 폴더에 favicon.ico 파일을 추가합니다.\n3. Spring Boot의 application.properties에 spring.web.resources.static-locations을 적절히 설정합니다.\n4. 필요 시 WebMvcConfigurer를 구현해 addResourceHandlers 메서드에서 favicon.ico를 매핑합니다.\n5. 테스트 환경에서 브라우저가 favicon.ico를 정상적으로 요청하고 응답되는지 확인합니다.',
 'keywords': 'favicon.ico 요청에 대한 핸들러 미설정 NoHandlerFoundException No handler found for GET /favicon.ico',
 'vector': [-0.060760498046875,
  -0.052001953125,
  -0.060699462890625,
  0.02288818359375,
  -0.01483917236328125,
  -0.035064697265